# Fig. 4 | Functional diversification of microglia and macrophages across the human lifespan

**Panel A:** UMAP visualizations of microglial and macrophage functional subtypes.  
**Panel B:** Functional-subtype composition across developmental stages.  
**Panel C:** Regional enrichment of individual microglial and macrophage cell types.  
**Panel D:** Developmental trajectories of microglial and macrophage WGCNA modules.


The plotting environment provides annotated microglial and macrophage Seurat objects (`mic_object` and `mac_object`) and their corresponding hdWGCNA objects (`mic_wgcna` and `mac_wgcna`). Objects contain functional-subtype, developmental-stage, cell-type and brain-region annotations.


In [ ]:
%%R
library(Seurat)
library(hdWGCNA)
library(dplyr)
library(tidyr)
library(ggplot2)
library(patchwork)
library(ComplexHeatmap)


In [ ]:
%%R
function_cluster_colors <- c(
  'Proliferative' = '#F8AFB3', 'Enhanced redox' = '#00AEEF',
  'Inflammatory' = '#4474BA', 'Lipid associated' = '#E4E07A',
  'Surveilling' = '#B2509E', 'Reacting' = '#009444',
  'Synaptic interacting' = '#E76F51',
  'Neurodegenerative associated' = '#A97BBB',
  'Stress responsive' = '#E2B956'
)


## Panel A | Functional-subtype UMAPs


In [ ]:
%%R
p_mic_umap <- DimPlot(
  mic_object, reduction = 'umap', group.by = 'function_cluster',
  cols = function_cluster_colors, raster = TRUE
) + ggtitle('Microglia')

p_mac_umap <- DimPlot(
  mac_object, reduction = 'umap', group.by = 'function_cluster',
  cols = function_cluster_colors, raster = TRUE
) + ggtitle('Macrophage')

p_mic_umap + p_mac_umap + plot_layout(guides = 'collect') &
  theme(legend.position = 'bottom')


## Panel B | Developmental distributions of functional subtypes


In [ ]:
%%R
stage_distribution <- bind_rows(
  mic_object[[]] %>% mutate(cell_population = 'Microglia'),
  mac_object[[]] %>% mutate(cell_population = 'Macrophage')
) %>%
  count(cell_population, stage, function_cluster, name = 'n') %>%
  group_by(cell_population, stage) %>%
  mutate(proportion = n / sum(n) * 100) %>%
  ungroup()

ggplot(stage_distribution, aes(stage, proportion, fill = function_cluster)) +
  geom_col(width = 0.8) +
  facet_grid(cell_population ~ ., scales = 'free_y') +
  scale_fill_manual(values = function_cluster_colors) +
  labs(x = 'Developmental stage', y = 'Proportion within cell type (%)',
       fill = 'Functional subtype') +
  theme_classic()


## Panel C | Brain-region enrichment

Individual cell types are tested for enrichment across Level 2 brain regions using one-sided Fisher's exact tests. The heatmap is annotated by functional subtype and displays the rescaled log odds ratio.


In [ ]:
%%R
mic_region_enrichment <- calculate_region_enrichment(mic_object, cell_type_col = 'type_3', region_col = 'region_L2')
mac_region_enrichment <- calculate_region_enrichment(mac_object, cell_type_col = 'type_3', region_col = 'region_L2')

plot_region_enrichment(
  microglia = mic_region_enrichment, macrophage = mac_region_enrichment,
  subtype_col = 'function_cluster', value_col = 'enrichment_score'
)


## Panel D | WGCNA module trajectories

Module eigengenes are plotted across developmental stages to identify modules that peak in adulthood, show embryonic and middle-to-late adult activity, or decline from embryonic levels.


In [ ]:
%%R
p_mic_modules <- PlotModuleTrajectory(mic_wgcna, pseudotime_col = 'stage_num', ncol = 2) +
  ggtitle('Microglia')
p_mac_modules <- PlotModuleTrajectory(mac_wgcna, pseudotime_col = 'stage_num', ncol = 2) +
  ggtitle('Macrophage')

p_mic_modules / p_mac_modules
